### Day 16 - Pandas 基础

**引入:Excel 表格比数组更直观**

抽问上节 (NumPy): `np.array(["张三", 20, 85.5]).dtype` 是什么? `(<U11`,全转成字符串)。NumPy 数组同构,无法保存"姓名+年龄+成绩"的表格
数据。Pandas 的 DataFrame = "NumPy 数组 + 行标签 + 列标签",正是 Excel 的样子。

> **今日目标**
> - 理解 Series(一维标签数组)和 DataFrame(二维标签表)
> - 会用 head/info/describe 快速检查数据
> - 会用 loc/iloc/布尔索引筛选行列
> - 会处理缺失值、排序

**前置**:Day14-15 已学 NumPy 数组。本节所有代码假设已 `import pandas as pd`。

#### 为什么需要 Pandas -- NumPy 没有的,才是痛点

> **痛点**:NumPy 数组元素类型一致,做"姓名+年龄+成绩"这种混合表格时,要么全变字符串(丧失数值运算),要么用多个数组(列关系靠人工记忆)。
>
> **类比**:NumPy 像一摞资料整齐捆着但不分类,Pandas 像 Excel 档案柜,每列有名字,每行有编号,一眼看懂。
>
> **解释**:Pandas 在 NumPy 之上加了索引层,支持异构列、标签访问、缺失值处理、数据清洗 -- 做数据分析的标配。

> **ASCII 对比**
> ```
> NumPy 数组:                    Pandas DataFrame:
> [张三 20 85.5]                  name  age  score
> [李四 21 92.0]                  ----  ---  -----
> [王五 19 78.5]                  张三   20   85.5
>  ^ 列含义靠注释                李四   21   92.0
>                                王五   19   78.5
>                                ^ 列名就是标签
> ```

In [ ]:
import pandas as pd

# 创建一个小 DataFrame:3 行 3 列
df = pd.DataFrame({
    "name": ["张三", "李四", "王五"],
    "age": [20, 21, 19],
    "score": [85.5, 92.0, 78.5]
})
print(df)
print("---")
print(df.dtypes)

# --- 执行过程 ---
# 第 2-5 行 pd.DataFrame({...}):
#   1. 接收一个 dict,键=列名,值=列数据(列表)
#   2. 三列长度都是 3,合法 -> 创建 3x3 二维表
#   3. 自动加上行索引 [0, 1, 2]
#   4. df 变量绑定到这个 DataFrame 对象
# 第 6 行 print(df):
#   1. 调用 df.__repr__(),按行列格式渲染
#   2. 列名(表头)在上,行索引在左,数据居中
# 第 8 行 print(df.dtypes):
#   1. 输出每列的数据类型
#   2. name(object 即 str),age(int64),score(float64)
#   3. 证明列间异构 -- 正是 NumPy 做不到的

> **逐行解剖**
> - `pd.DataFrame(dict)`:接受 dict,键=列名,值=一等长列表
> - `df.dtypes`:返回每列的 NumPy dtype(int64/float64/object 等)
> - 自动索引:不指定时默认 0..n-1

> **常见错误**
> 1. **错误现象**:`ValueError: All arrays must be of the same length`
>    **原因**:三个列表长度不一致
> 2. **错误现象**:`import pandas as pd` 写错成 `import pandas`
>    **原因**:必须带 `as pd`,这是业界标准写法
> 3. **错误现象**:`NameError: name 'pd' is not defined`
>    **原因**:跳到下一个 cell 运行,但忘记前面先运行 `import pandas as pd` 那个 cell

> 问自己:
> - 如果某天"姓名"列混入了数字,DataFrame 还能正常创建吗?
> - df.shape 返回的是 (行数,列数) 还是 (列数,行数)?
> - 为什么 Pandas 里每列一种 dtype,而整个表不是单一 dtype?

In [ ]:
import pandas as pd

# ============ 学员代码区 ============
# 创建一个 DataFrame,列: city, population(万人),province
# 填入 3 个真实城市的数据
df_city = pd.DataFrame({
    "city": [],
    "population": [],
    "province": []
})
print(df_city)
print(df_city.dtypes)

# ============ 参考答案 ============
df_city = pd.DataFrame({
    "city": ["北京", "上海", "广州"],
    "population": [2184, 2487, 1882],
    "province": ["北京", "上海", "广东"]
})
print(df_city)
print(df_city.dtypes)

#### Series 创建 -- 带标签的一维数组

> **痛点**:NumPy 一维数组 `arr[2]` 数字索引含义不明确,下标 2 到底指\u201c第三条记录\u201d还是\u201c第二个字段\u201d得靠注释猜。
>
> **类比**:Series = Excel 里的单列,**既有值又有标签**,标签可以是字符串。
>
> **解释**:`pd.Series(data, index=...)` 给每个元素挂上标签,访问时既可按位置,也可按标签。

> **ASCII 内存图**
> ```
> NumPy:              pd.Series:
> arr = [10 20 30]    s = pd.Series([10, 20, 30],
>                     index=["a", "b", "c"])
>                     +---+----+
> arr[1] -> 20       | a | 10 | <- s["a"] -> 10
>                     +---+----+
>                     | b | 20 | <- s[1]   -> 20
>                     +---+----+
>                     | c | 30 | <- s["c"] -> 30
>                     +---+----+
>                     标签与值一一绑定
> ```

In [ ]:
import pandas as pd

s1 = pd.Series([10, 20, 30])
print("默认索引:")
print(s1)

s2 = pd.Series([10, 20, 30], index=["a", "b", "c"])
print("\n自定义索引:")
print(s2)
print("访问 b:", s2["b"])
print("访问第二个:", s2.iloc[1])

# --- 执行过程 ---
# 第 2 行:
#   1. 接收列表 [10, 20, 30]
#   2. 没给 index 参数,自动分配 0, 1, 2
#   3. 返回新 Series 对象
# 第 6 行:
#   1. index 列表长度与 data 长度一致(都是 3)
#   2. 把 10 绑到 "a",20 绑到 "b",30 绑到 "c"
#   3. 返回新 Series
# 第 9 行 s2["b"]:
#   1. 按标签访问 -- Pandas 自动在 index 里找 "b"
#   2. 找到后返回对应的值 20
# 第 10 行 s2.iloc[1]:
#   1. iloc 严格按位置访问
#   2. 位置 1 = 20(不受标签类型影响)

> **逐行解剖**
> - `pd.Series(data, index=...)`:data 可以是列表、NumPy 数组、dict
> - 优先标签:s[整数] 先查 index,查不到再按位置
> - 自定义 index 长度必须与 data 长度一致

> **常见错误**
> 1. **错误现象**:`ValueError: Length of values (3) does not match length of index (2)`
>    **原因**:data 有 3 个元素,index 只写 2 个
> 2. **错误现象**:拿到 None 而不是数字
>    **原因**:用 s[0] 访问自定义整数索引 -- 标签优先,找不到就报错
> 3. **错误现象**:期望按位置却按了标签
> 4. **原因**:混用位置与标签导致混淆,严格区分 loc(标签)/iloc(位置)

> 问自己:
> - 如果自定义 index 与位置冲突,结果怎样?
> - Series 和 Python 列表最大的区别在哪里?
> - 为什么分析单列常用 Series,多列用 DataFrame?

In [ ]:
import pandas as pd

# ============ 学员代码区 ============
# 创建 1 个 Series:3 个城市的平均气温
# 值:[25.5, 30.1, 28.7]
# 标签:["北京", "上海", "广州"]
# 1. 打印整个 Series
# 2. 用标签访问 "上海" 的温度
# 3. 用位置访问第三个城市的温度
pass

# ============ 参考答案 ============
temps = pd.Series(
    [25.5, 30.1, 28.7],
    index=["北京", "上海", "广州"]
)
print("全部数据:")
print(temps)
print("\n上海:", temps["上海"])
print("第三个:", temps.iloc[2])

#### DataFrame 创建 -- 从字典组装表格

> **痛点**:需要一张二维表,NumPy 数组列之间没有名字,新增列很容易弄混。
>
> **类比**:DataFrame 就像 Excel:\u201c列名=表头,列值=列数据\u201d;从字典创建,键=列名,值=这一列的列表。
>
> **解释**:`pd.DataFrame(dict)` 是最常用的方式,dict 的每个值必须等长。

In [ ]:
import pandas as pd

data = {
    "name": ["张三", "李四", "王五"],
    "age": [20, 21, 19],
    "score": [85.5, 92.0, 78.5]
}
df = pd.DataFrame(data)
print(df)
print("\n行索引:", list(df.index))
print("列名:", list(df.columns))

# --- 执行过程 ---
# 第 2-5 行 data = {...}:
#   1. 建 3 个键值对,每个 key 为列名
#   2. 每个 value 为等长列表(3 个元素)
# 第 8 行 pd.DataFrame(data):
#   1. 读取 dict,每个 key 变成 df 的一列
#   2. 默认 index = [0, 1, 2]
#   3. 返回 DataFrame 对象
# 第 10 行 df.index:
#   1. 取行标签,默认 RangeIndex(0, 1, 2)
# 第 11 行 df.columns:
#   1. 取列名 Index(["name", "age", "score"])

> **逐行解剖**
> - `dict.keys()` 决定列名,`dict.values()` 决定列数据
> - 创建后 `df.index` / `df.columns` 可独立查看
> - 各列 dtype 独立(name=object,age=int,score=float)

> **常见错误**
> 1. **错误现象**:`ValueError: arrays must all be same length`
>    **原因**:某个键的值长度不等
> 2. **错误现象**:列顺序不稳定
>    **原因**:Python 3.6 之前 dict 无序;如需保序,用 `columns=` 参数显式指定
> 3. **错误现象**:键名写错导致新增一列
>    **原因**:{"nme":[...]} 比 {"name":[...]} 多一列 nme,原 name 列缺失

> 问自己:
> - 如果值列表为空 {"name":[]},df.shape 是多少?
> - 用 `pd.DataFrame(data, columns=['score','age'])` 会怎样?列呢?
> - 为什么大规模分析都用 DataFrame 而不是嵌套列表?

In [ ]:
import pandas as pd

# ============ 学员代码区 ============
# 用字典创建 4 行 3 列的 DataFrame:
# - city:["北京", "上海", "广州", "深圳"]
# - population:[2184, 2487, 1882, 1779] (万人)
# - province:["北京", "上海", "广东", "广东"]
# 1. 创建 df
# 2. 打印它的 shape
# 3. 打印列名列表
pass

# ============ 参考答案 ============
city_df = pd.DataFrame({
    "city": ["北京", "上海", "广州", "深圳"],
    "population": [2184, 2487, 1882, 1779],
    "province": ["北京", "上海", "广东", "广东"]
})
print(city_df)
print("shape:", city_df.shape)
print("列名:", list(city_df.columns))

#### 数据查看 -- 先看轮廓,再深入细节

> **痛点**:数据表几十万行,print 全部看不懂;先得知道"有多少行列、每列啥类型、是否有缺失"。
>
> **类比**:像刚接手一本档案,先看目录(info)、看首尾几页(head/tail)、翻统计摘要(describe),再决定怎么读。
>
> **解释**:`head(n)` / `tail(n)` / `info()` / `describe()` 是高频 EDA 四件套。

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "name": ["张三", "李四", "王五", "赵六", "钱七"],
    "age": [20, 21, 19, 22, 20],
    "score": [85.5, 92.0, None, 88.0, 76.5]
})

print("=== df.head(2) ===")
print(df.head(2))
print("\n=== df.tail(2) ===")
print(df.tail(2))
print("\nshape:", df.shape)

# --- 执行过程 ---
# 第 1 行:创建 DataFrame,None 显示为 NaN
# 第 8 行 df.head(2):
#   1. 取行 0..1,返回新 DataFrame
#   2. head() 是预读,不修改 df
# 第 11 行 df.tail(2):
#   1. 取最后 2 行(index = 3, 4)
# 第 13 行 df.shape:
#   1. 返回元组 (行数,列数) = (5, 3)

> **逐行解剖**
> - `head(n=5)` / `tail(n=5)`:取首/尾,不修改原 df
> - `df.shape`:元组 (行,列),len(df) 也能拿行数
> - `df.size`:总单元格数,`df.ndim`:维度(恒为 2)

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "name": ["张三", "李四", "王五", "赵六", "钱七"],
    "age": [20, 21, 19, 22, 20],
    "score": [85.5, 92.0, None, 88.0, 76.5]
})

print("=== df.info() ===")
df.info()
print("\n=== df.describe() ===")
print(df.describe())

# --- 执行过程 ---
# 第 8 行 df.info():
#   1. 打印列名、非空数、dtype
#   2. score 列显示 Non-Null Count = 4(1 个缺失)
# 第 11 行 df.describe():
#   1. 只统计数值列(age,score),跳过 name 这种 object
#   2. count/mean/std/min/25%/50%/75%/max 八项
#   3. 这里是 count = 4(缺失不算)

> **逐行解剖**
> - `df.info()`:列名+非空数+内存查看,必跑,随时发现缺失
> - `df.describe()`:只统计数值列,包含四分位数
> - `df.describe(include="object")` 可统计非数值列

> **常见错误**
> 1. **错误现象**:info() 没有一个 print 包就被静默了
>    **原因**:info() 直接打印,不是返回值,`x = df.info()` 得到 None
> 2. **错误现象**:describe() 没显示 name 列
>    **原因**:describe 默认只看数值列,非数值要加 `include='object'`
> 3. **错误现象**:以为 head() 会修改 dataframe
>    **原因**:head/tail 是 view/copy,不是 in-place

> 问自己:
> - 如果某列 count 小于总行数(在 info 里),意味着什么?
> - describe() 的 50% 和中位数有什么关系?
> - 为什么 head(n) 的 n 通常取 5 而不是 500?

In [ ]:
import pandas as pd

# ============ 学员代码区 ============
# 创建 DataFrame:6 行,3 列(A:int,B:float,C:str),
# 其中一行的 B 值为 None
# 1. 使用 head(3)/tail(2)/info()/describe()
# 2. 输出 df.shape
pass

# ============ 参考答案 ============
df_ex = pd.DataFrame({
    "A": [1, 2, 3, 4, 5, 6],
    "B": [1.1, 2.2, None, 4.4, 5.5, 6.6],
    "C": ["a", "b", "c", "d", "e", "f"]
})
print(df_ex.head(3))
print(df_ex.tail(2))
print("shape:", df_ex.shape)
df_ex.info()
df_ex.describe()

#### loc 与 iloc -- 标签 vs 位置

> **痛点**:有时按\u201c学生名字\u201d找(标签),有时按\u201c第 3 行\u201d找(位置),混淆后常报错。
>
> **类比**:`loc` 是\u201c按名字找人\u201d(图书馆按书名找),`iloc` 是\u201c按编号找\u201d(书架号找书)。
>
> **解释**:
> - `df.loc[标签]` 按 index 值 + 列名,**包含右端**
> - `df.iloc[位置]` 按整数下标,**不含右端**(同 Python 切片)

> **ASCII 对照**
> ```
> label = ["a", "b", "c", "d"]
> pos   =   0   1   2   3
> df.loc["b":"d"]  -> 行 b, c, d (含 d)
> df.iloc[1:3]       -> 行 1, 2    (不含 3)
> 口诀: loc 闭, iloc 开
> ```

In [ ]:
import pandas as pd

df = pd.DataFrame(
    {"X": [1, 2, 3, 4], "Y": [10, 20, 30, 40]},
    index=["a", "b", "c", "d"]
)
# --- loc 标签选 ---
print("=== loc ===")
print(df.loc["b"])
print()
print(df.loc["b":"c"])

# --- iloc 位置选 ---
print("\n=== iloc ===")
print(df.iloc[0])
print()
print(df.iloc[1:3])

# --- 同时选行列 ---
print("\n=== loc + 列 ===")
print(df.loc["b":"d", ["X"]])

# --- 执行过程 ---
# 第 1 行:建 4x2 表,行标 a/b/c/d
# 第 6 行 df.loc["b"]:
#   1. 标签访问 -> 返回 X=2, Y=20 的 Series
# 第 8 行 df.loc["b":"c"]:
#   1. 标签切片 -> 右端包含
#   2. 返回 b 和 c 两行,列不变
# 第 12 行 df.iloc[0]:
#   1. 位置访问 -> 第 0 行(就是 a 行)
#   2. 与 df.loc['a'] 结果一样,但机制不同
# 第 14 行 df.iloc[1:3]:
#   1. 整数切片 -> 不含右端
#   2. 返回位置 1 和 2(即 b, c 两行)

> **逐行解剖**
> - `df.loc[row,col]`:两端闭,标签访问
> - `df.iloc[row,col]`:左闭右开,位置访问
> - `df.loc[:, "X"]` 冒号 = 全选
> - `df['X']` 是列访问,不是行访问

> **常见错误**
> 1. **错误现象**:`KeyError: 0`(在 loc 中用 0)
>    **原因**:loc 用标签索引,0 不在 index 里;改用 iloc[0]
> 2. **错误现象**:loc 切片忘了右闭
>    **原因**:习惯 Python 右开,误以为 loc[1:3] 不含 3
> 3. **错误现象**:`df["X"]` 以为选了第 X 行
>    **原因**:df[字符串] 是按**列名**选列,不是选行

> 问自己:
> - 如果想同时选第 2-3 行和前两列,用 iloc 怎么写?
> - 如果标签恰好是整数 `[10,20,30]`,`df.loc[10]` 还是标签还是位置?
> - 为什么说 `df["name"]` 是列访问?有哪些等价写法?

In [ ]:
import pandas as pd

# ============ 学员代码区 ============
# df = pd.DataFrame(A:[1,2,3,4,5],B:[10,20,30,40,50],
#                   index:["a","b","c","d","e"])
# 1. 用 loc 选 b 到 d 行
# 2. 用 iloc 选前三行
# 3. 用 iloc 选第 2 到 4 行(位置 2,3,4)
# 4. 同时选 b..d 行与 A 列
pass

# ============ 参考答案 ============
df_test = pd.DataFrame(
    {"A": [1, 2, 3, 4, 5], "B": [10, 20, 30, 40, 50]},
    index=["a", "b", "c", "d", "e"]
)
print(df_test.loc["b":"d"])
print(df_test.iloc[0:3])
print(df_test.iloc[2:5])
print(df_test.loc["b":"d", "A"])

#### 布尔索引 -- 用条件筛选行

> **痛点**:只关心\u201c成绩高于 90\u201d或\u201c年龄小于 20\u201d的部分数据,不想手动翻找。
>
> **类比**:像 Excel 里设筛选条件 -- Pandas 的布尔索引就是\u201c每行 True/False 打分,只留 True 的行\u201d。
>
> **解释**:单列比较返回 True/False 的 Series,放进 df[...] 即可筛选;多条件用 `&`|`|`~`,每个条件必须加括号

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "name": ["张三", "李四", "王五", "赵六", "钱七"],
    "age": [20, 21, 19, 22, 20],
    "score": [85.5, 92.0, 78.5, 88.0, 76.5]
})
mask1 = df["score"] >= 85
print("score >= 85 的 mask:")
print(mask1)
print("\n筛选结果:")
print(df[mask1])

mask2 = (df["score"] >= 80) & (df["age"] <= 20)
print("\nscore >= 80 且 age <= 20:")
print(df[mask2])

# --- 执行过程 ---
# 第 6 行 df["score"] >= 85:
#   1. 对 score 列逐行比较 -> 返回 bool Series
#   2. [True,True,False,True,False]
# 第 9 行 df[mask1]:
#   1. Pandas 布尔索引:保留 mask=True 的行
#   2. 返回新 df(不修改原 df)
# 第 11 行 (...)&(...):
#   1. 每个条件返回 bool Series
#   2. & 对应位做与运算 -> 新 bool Series
#   3. 括号!去掉括号会报优先级错误

> **逐行解剖**
> - `df[mask]`:mask 是长度等于 df 行数的 bool Series
> - `&` 且 / `|` 或 / `~` 非,每个条件必须加括号
> - `df["col"].isin([v1,v2])`:等价于多 or

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "name": ["张三", "李四", "王五", "赵六", "钱七"],
    "age": [20, 21, 19, 22, 20],
    "score": [85.5, 92.0, 78.5, 88.0, 76.5]
})
target_names = ["张三", "赵六"]
print(df[df["name"].isin(target_names)])

# --- 执行过程 ---
# 第 6 行 df["name"].isin(target_names):
#   1. 对 name 列逐行:"张三" in list? -> True
#   2. 返回 bool Series
#   3. = [True, False, False, True, False]
# 第 7 行 df[...]:保留 True 的行

> **常见错误**
> 1. **错误现象**:`ValueError: The truth value of a Series is ambiguous`
>    **原因**:Python 用 `and`/`or` 连 Pandas Series,要改成 `&`/`|`
> 2. **错误现象**:落了括号,优先级错乱
>    **原因**:`df['A']>1 & df['B']<2` 等于
>    `df['A']>(1 & df['B'])<2`
> 3. **错误现象**:isin 写成 `df["A"] in [1,2,3]`
>    **原因**:Python in 一次判单值,Series 用 isin
> 问自己:
> - `df[mask]` 中 mask 的长度必须等于什么?
> - `.isin(["a","b"])` 等价于哪些 or 条件组合?
> - 想取 score<80 或 age>21 两条件的并集,

In [ ]:
import pandas as pd

# ============ 学员代码区 ============
# df = PD 4 行:city,population
#    index:[BJ,SH,GZ,SZ]
# 1. 用布尔索引选人口 > 1800 的城市
# 2. 用 isin 同时选 BJ 和 GZ 两行
# 3. 用 ~ 反向筛选:选人口 <= 2000 的城市
pass

# ============ 参考答案 ============
df_city = pd.DataFrame(
    {"city": ["北京", "上海", "广州", "深圳"],
     "population": [2184, 2487, 1882, 1779]},
    index=["BJ", "SH", "GZ", "SZ"]
)
print("大过 1800:")
print(df_city[df_city["population"] > 1800])
print("\nisin BJ/GZ:")
print(df_city[df_city.index.isin(["BJ", "GZ"])])
print("\n不大于 2000:")
print(df_city[~(df_city["population"] > 2000)])

#### 缺失值处理 -- dropna / fillna

> **痛点**:现实数据常见缺失, NaN 会\u201c污染\u201d统计(比如 `mean()` 遇到 NaN 变 NaN)。
>
> **类比**:像档案里某页空了 -- 要么整卷扔掉(dropna),要么用前后页内容补抄(fillna)。
>
> **解释**:
> - `dropna()`:删除含 NaN 的行(或列)
> - `fillna(value)`:用固定值、均值、前向填充等补 NaN

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "name": ["张三", "李四", "王五", "赵六"],
    "age": [20, 21, None, 22],
    "score": [85.5, None, 78.5, 88.0]
})
print("=== 原始 ===")
print(df)
print("\n缺失统计:")
print(df.isna().sum())
print("\n=== dropna ===")
print(df.dropna())
print("\n=== fillna(0) ===")
print(df.fillna(0))

# --- 执行过程 ---
# 第 4 行 None:
#   1. Pandas 把 None 转为 np.nan(显示为 NaN)
#   2. 该列自动升为 float
# 第 8 行 df.isna().sum():
#   1. isna():逐元素判断 -> bool DataFrame
#   2. sum():按列求和(True=1,False=0)
# 第 10 行 df.dropna():
#   1. 默认 axis=0(删行):只要本行含 NaN 就整行删
#   2. 只剩 index = 3
# 第 12 行 df.fillna(0):
#   1. 用 0 替换所有 NaN,返回新 df

> **逐行解剖**
> - `df.isna()` / `df.isnull()`:逐元素 bool 判断
> - `df.dropna(axis=0, how='any')`:how='all' 只删**全 NaN** 的行
> - `df.fillna(method='ffill')`:前向填充
> - 默认返回**新对象**,不修改原 df

> **常见错误**
> 1. **错误现象**:以为 `df.fillna(0)` 修改了 df,实际原样没变
>    **原因**:默认返回新对象;要 in-place 需 `df.fillna(0, inplace=True)`
> 2. **错误现象**:dropna 后行号不连续
>    **原因**:drop 删除行,index 不重建;必要时 `.reset_index(drop=True)`
> 3. **错误现象**:写 `if age == None` 判空报错
>    **原因**:Pandas 缺失值是 NaN,NaN != NaN;要用 `pd.isna(age)`

> 问自己:
> - dropna(how='all') 和 dropna() 有什么区别?
> - 填均值而不是 0 的好处是?
> - 为什么 fillna 默认不修改原 df?

In [ ]:
import pandas as pd

# ============ 学员代码区 ============
# df = pd.DataFrame({"A":[1,None,3,4,5],
#                    "B":[10,20,30,None,50]},
#                   index=["a","b","c","d","e"])
# 1. 用 isna().sum() 统计每列缺失数
# 2. 用 dropna(how="all") 删全 NaN 行
# 3. 用 fillna(某列均值) 补 NaN
# 4. 用 ffill 前向填充
pass

# ============ 参考答案 ============
df_ex = pd.DataFrame(
    {"A": [1, None, 3, 4, 5], "B": [10, 20, 30, None, 50]},
    index=["a", "b", "c", "d", "e"]
)
print("缺失数:\n", df_ex.isna().sum(), sep="")
print("\ndropna(all):\n", df_ex.dropna(how="all"), sep="")
print("\nfillna(均值):\n", df_ex.fillna(df_ex.mean()), sep="")
print("\nffill:\n", df_ex.ffill(), sep="")

#### 数据排序 -- sort_values / sort_index

> **痛点**:只看\u201c成绩最高 Top3\u201d或\u201c按省份排序统计\u201d,原始 df 没有顺序。
>
> **类比**:像图书馆重新按索书号(sort_values)或入库时间(sort_index)整理,让查阅方便。
>
> **解释**:
> - `sort_values("列名")`:按某列升序/降序
> - `sort_index()`:按行标签排序

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "name": ["张三", "李四", "王五", "赵六"],
    "age": [20, 21, 19, 22],
    "score": [85.5, 92.0, 78.5, 88.0]
})
asc = df.sort_values("score", ascending=False)
print("=== score 降序 ===")
print(asc)
multi = df.sort_values(
    ["age", "score"], ascending=[True, False]
)
print("\n=== age 升 + score 降 ===")
print(multi)

# --- 执行过程 ---
# 第 7 行 df.sort_values("score",ascending=False):
#   1. 按 score 列值排序,从高到低
#   2. 返回新 df,index 跟着移动
# 第 11 行 df.sort_values(..., ascending=[...]):
#   1. 多列排序:先按 age 升序
#   2. age 相同时按 score 降序
#   3. ascending 是列表,与 by 一一对应

> **逐行解剖**
> - `sort_values(by, ascending=True)`:by 可以是列名或列表
> - `sort_index(axis=0)`:axis=1 时按列名排序
> - NaN 默认排最后(`na_position='last'`)

In [ ]:
import pandas as pd

df2 = pd.DataFrame(
    {"val": [30, 10, 20]},
    index=["c", "a", "b"]
)
print("=== 原始 ===")
print(df2)
print("\n=== sort_index 后 ===")
print(df2.sort_index())

# --- 执行过程 ---
# 第 2 行:创建 df2,index 乱序 c, a, b
# 第 6 行 df2.sort_index():
#   1. 按 index 字母升序 -> a, b, c
#   2. 数据跟着重排

> **常见错误**
> 1. **错误现象**:sort_values 后 df 没变
>    **原因**:返回新 df;需 `df = df.sort_values(...)` 或 `inplace=True`
> 2. **错误现象**:多列排序漏了 ascending 列表
>    **原因**:`ascending=[True,False]` 长度必须与 by 一致
> 3. **错误现象**:传错 na_position
>    **原因**:默认 `last`;若想排最前写 `na_position='first'`
> 问自己:
> - sort_values(by=["A","B"]) 中等级怎么体现?
> - reset_index(drop=True) 通常在什么时候用?
> - 为什么读取 CSV 后经常要先 sort_index?

In [ ]:
import pandas as pd

# ============ 学员代码区 ============
# df = PD({"name":[3 人],"score":[3 个],"class":[3 个]})
# 1. 按 score 降序排列,取 Series (name)
# 2. 按 class 升序,再按 score 降序
# 3. 排序后重置 index (drop=True)
pass

# ============ 参考答案 ============
df_ex = pd.DataFrame({
    "name": ["张三", "李四", "王五"],
    "score": [85, 92, 78],
    "class": ["A", "B", "A"]
})
print("score 降序 name:\n",
      df_ex.sort_values("score", ascending=False)["name"],
      sep="")
print("\nclass 升 + score 降:\n",
      df_ex.sort_values(["class", "score"],
                        ascending=[True, False]), sep="")
print("\n重置 index:\n",
      df_ex.reset_index(drop=True), sep="")

#### NCDL 负案例 -- 故意写错,看看报错什么样

> **场景**:"高分榜"应该取 score >= 90,但新手漏写括号
```python
high = df[df["score"] >= 90 & df["age"] < 22]
```这会报 ValueError。先演示错误,再给正确解法

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "name": ["张三", "李四", "王五", "赵六"],
    "age": [20, 21, 19, 22],
    "score": [85.5, 92.0, 78.5, 88.0]
})

# ============ BREAK IT 演示 ============
try:
    # 错误写法:score >= 90 & age < 22
    # 实际被解释为:score >= (90 & age) < 22
    wrong = df[df["score"] >= 90 & df["age"] < 22]
except ValueError as e:
    print("报错:", e)
    print("原因:& 优先级比 >= 高,需要括号!")
# ============ END BREAK IT ============

correct = df[(df["score"] >= 90) & (df["age"] < 22)]
print("\n正确结果:")
print(correct)

# --- 执行过程 ---
# 第 9 行(错误代码):
#   1. & 是 Python 位运算,优先级比 >= 高
#   2. 90 & df["age"] 先算,结果 int Series
#   3. 再 df['score'] >= int_Series 报错
# 第 15 行(正确代码):
#   1. 括号内 df["score"] >= 90 求值 -> bool
#   2. & 两 bool Series -> 新 bool Series
#   3. df[bool_Series] 筛选

> **逐行解剖**
> - `&` 是**位运算**,优先级**高于**比较运算符
> - 正确的做法:每个比较都加括号
> - 用 try/except 捕获预期错误,让 Break It 安全落地

#### 综合练习 -- 建表 + 查看 + 过滤 + 排序

综合今日 8 道知识点,一个完整小项目。

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "name": ["张三", "李四", "王五", "赵六", "钱七"],
    "age": [20, 21, 19, 22, 20],
    "score": [85.5, 92.0, None, 88.0, 76.5],
    "class": ["A", "B", "A", "B", "A"]
})

# 1. 查看概览
print(df.info())
print(df.describe())

# 2. 过滤:score >= 85
high_score = df[df["score"] >= 85]
print("\n=== score >= 85 ===")
print(high_score)

# 3. 用布尔索引 + loc 同时选行和列
print("\n=== 张三的成绩 ===")
print(df.loc[df["name"] == "张三", ["name", "score"]])

# 4. 排序:score 降序,填充缺失值后再排
df_filled = df.fillna(df["score"].mean())
print("\n=== 填均值后 sort ===")
print(df_filled.sort_values('score', ascending=False))

# --- 执行过程 ---
# 第 8 行 df.info():
#   1. 快速扫表,score Non-Null Count = 4(1 个缺失)
# 第 11 行 df["score"] >= 85:
#   1. 返回 bool Series,NaN 行返回 False
# 第 15 行 df.loc[bool, ["name","score"]]:
#   1. 布尔选行 + 列名列表选列
#   2. 同时筛选行和列
# 第 19 行 df["score"].mean():
#   1. 自动跳过 NaN,计算非空值均值
#   2. 常用填缺策略

> **逐行解剖**
> - 实际项目中,info/describe 必跑第一遍
> - fillna 常用:`df.mean()`/`df.ffill()`
> - 缺失值行在布尔筛选中跳过(NaN 返回 False)

**今日小结**

| 概念 | 解决的痛点 |
| ------------ | ------------ |
| 为什么需要 Pandas | NumPy 数组异构表格难处理 |
| Series | 带标签一维数组 |
| DataFrame 从字典建 | 二维表,每列可不同类型 |
| head/tail/shape | 首尾快速预览 |
| info/describe | 概览发现缺失 |
| loc/iloc | 标签 vs 位置选行 |
| 布尔索引 & ~ | 按条件筛选 |
| dropna/fillna | 缺失处理 |
| sort_values/sort_index | 排序查看 |

> **8 要点回顾**
> - 创建:Series(data,index);DataFrame(dict)
> - 查看:head/tail/info/describe/shape/dtypes
> - 选取:loc(标签闭区间),iloc(位置开区间)
> - 过滤:bool Series + &/|/~
> - 缺失:dropna(删)/fillna(填)
> - 排序:sort_values/sort_index

**更多练习**

- 当堂练:course/lesson13/in_class/practice01-06.py
- 课后作业:course/lesson13/homework/task01-03.py